In [6]:
# torch, faiss (k-NN/perplexity graph construction) and PHATE's numba-based internals
# all get loaded into one process here; with unconstrained thread pools that combination
# makes macOS abort with a duplicate-OpenMP-runtime error (kernel crash / segfault).
# Capping thread counts before any of those libraries import, plus the (unsafe but
# standard) escape hatch as a second line of defense, avoids it.
import os
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('NUMEXPR_NUM_THREADS', '1')
os.environ.setdefault('NUMBA_NUM_THREADS', '1')

import sys
import json
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import napari
from scipy.interpolate import griddata
from pathlib import Path

from plot_utils_era5 import *

project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root / 'src'))
sys.path.insert(0, str(Path.cwd().parent / 'scripts'))

sys.path.append(f"{project_root}src/")
# Import utilities
from utilities import KernelNN, KernelNN_subgraph, GraphLMDBReader, setup_device, get_output_affinities
from plot_utilities import plot_results_napari, plot_results_matplotlib

sys.path.append(f"{project_root}/ERA5/scripts/")
from utilities_era5 import ERA5_Graph_Generator as Data_Generator
from utilities_era5 import ERA5_Graph_Generator_perSampleNorm as Data_Generator_old
data_path = project_root / 'ERA5' / 'data'

# Device setup
device = setup_device()
print(f'Using device: {device}')

#from sklearn.decomposition import PCA
#from sklearn.manifold import TSNE
import subprocess
import phate
import anndata as ad
import concord as ccd
import synthia as syn

from scipy.spatial import cKDTree

def load_model(id:str, model_type:str='random_crop'):
    model_path = project_root / 'ERA5' / 'trained_models' / model_type / f'{id}.pth'
    config_path = project_root / 'ERA5' / 'trained_models' / model_type / 'logs' / 'configs' / f'{id}.json' 
    
    with open(config_path, 'r') as f:
        config = json.load(f)
    
    model = KernelNN(
        width=config['width'],
        ker_width=config['ker_width'],
        depth=config['depth'],
        ker_in=config['edge_attr'],
        in_width=config['node_features'],
        out_width=config['out_width']
    ).to(device)
    
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    return model, config

Using device: mps


----
### load model + dataset 

In [7]:
# The trained model checkpoint/config for 'regular_papermodel2_2026-05-04_t12-01-43'
# isn't present in this local repo copy, and NOFE inference is disabled below anyway --
# use its known config values directly instead of load_model().
config = {
    'model_config_path': 'ERA5/configurations/regular/model_configs/papermodel2.json',
    'dataset_id': 'dataset_reg_fixcrop_median_woPos_tkernel',
    'dataset_path': 'ERA5/datasets/regular',
    'learning_rate': 1e-05,
    'scheduler_step': 10,
    'scheduler_gamma': 0.5,
    'epochs': 25,
    'random_seed': 42,
    'use_gradient_clipping': False,
    'track_gradients': True,
    'checkpoint_every': 5,
    'width': 64,
    'ker_width': 16,
    'depth': 3,
    'node_features': 11,
    'edge_attr': 7,
    'out_width': 3,
    'data_folder_path': '../data/',
    'dataset_save_path': '../datasets/regular/',
    'lat_min': 30,
    'lat_max': 75,
    'lon_min': -12,
    'lon_max': 33,
    'N_min': 500,
    'N_max': 10000,
    'k': 10,
    'perplexity': 10,
    'sampling_method': 'random',
    'train_date_start_end': ['2018-01-01', '2018-12-31'],
    'test_date_start_end': ['2019-01-01', '2019-01-31'],
    'train_samples_per_date': 10,
    'test_samples_per_date': 5,
    'feature_mode': 'channel_only',
    'sigma_method': 'global_median',
    'vars': ['w', 'v', 'z', 'q', 'r', 't', 'd', 'o3', 'pv', 'u', 'vo'],
}

In [8]:
lat_min = config["lat_min"]
lat_max = config["lat_max"]
lon_min = config["lon_min"]
lon_max = config["lon_max"]
#N = config["N"]
k = config["k"]
perplexity = config["perplexity"]
sampling_method = config["sampling_method"]
train_date_start_end = config["train_date_start_end"]
test_date_start_end = config["test_date_start_end"]
train_samples_per_date = config["train_samples_per_date"]
test_samples_per_date = config["test_samples_per_date"]
feature_mode = config["feature_mode"]
vars = config["vars"]


In [9]:
lat_range = (lat_max - lat_min)/2
lon_range = (lon_max - lon_min)/2
margin=5
side_crop=7
'''
bbox1 = lon_min, lat_min+lat_range-margin, lon_max-lon_range+margin, lat_max
bbox2 = lon_min+lon_range-margin, lat_min+lat_range-margin, lon_max, lat_max
bbox3 = lon_min, lat_min, lon_max-lon_range+margin, lat_max-lat_range+margin
bbox4 = lon_min+lon_range-margin, lat_min, lon_max, lat_max-lat_range+margin

bboxes = [bbox1, bbox2, bbox3, bbox4]
'''
bbox1 = lon_min+side_crop, lat_min+side_crop, lon_max-lon_range+margin, lat_max-side_crop
bbox2 = lon_min+lon_range-margin, lat_min+side_crop, lon_max-side_crop, lat_max-side_crop
bboxes = [bbox1, bbox2]

In [10]:
#nofe_mean_dist_per_pred = []
#pca_mean_dist_per_pred = []
#tsne_mean_dist_per_pred = []
#umap_mean_dist_per_pred = []
phate_mean_dist_per_pred = []
spatPCA_mean_dist_per_pred = []
concord_mean_dist_per_pred = []
fpca_mean_dist_per_pred = []


dates = pd.date_range("2019-01-01", "2019-01-31").strftime("%Y-%m-%d").tolist()

# Start the spatialPCA R process ONCE, outside the loop -- library(SpatialPCA) (Seurat,
# SPARK, etc.) is the slow part to load, and conda run's own activation overhead also
# used to get paid on every single call. It now stays alive and answers one request per
# line over stdin/stdout for the whole loop below.
spdir = f'{project_root}/ERA5/spatialpca'
# Unique per process so concurrently running scripts/notebooks don't collide on the
# same buffer filenames (a fixed call_id caused FileNotFoundError / silent data
# corruption when two spatialPCA calls raced on the same X.csv/coords.csv paths).
spatialpca_call_id = f'spatialpca_buffer_{os.getpid()}'
spatialpca_proc = subprocess.Popen(
    ['conda', 'run', '-n', 'spatialpca', '--no-capture-output', 'Rscript', 'execute_spatialpca_server.R'],
    cwd=spdir, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
    text=True, bufsize=1,
)
ready = spatialpca_proc.stdout.readline().strip()
if ready != 'READY':
    raise RuntimeError(f'spatialPCA server failed to start: {ready!r}\n{spatialpca_proc.stderr.read()}')
print('spatialPCA server ready.')


def spatialpca_embed(X, coords, call_id, n_components=3):
    np.savetxt(f'{spdir}/{call_id}_X.csv', X, delimiter=',')
    np.savetxt(f'{spdir}/{call_id}_coords.csv', coords, delimiter=',')

    spatialpca_proc.stdin.write(f'{call_id} {n_components}\n')
    spatialpca_proc.stdin.flush()
    response = spatialpca_proc.stdout.readline().strip()

    os.remove(f'{spdir}/{call_id}_X.csv')
    os.remove(f'{spdir}/{call_id}_coords.csv')

    if response != 'OK':
        raise RuntimeError(f'spatialPCA failed for {call_id}: {response}')

    Y = np.loadtxt(f'{spdir}/res_{call_id}.csv', delimiter=',', skiprows=1)
    os.remove(f'{spdir}/res_{call_id}.csv')
    return Y


def concord_embed(X, n_components=3):
    adata = ad.AnnData(X=X.astype(np.float32))
    cur_ccd = ccd.Concord(
        adata=adata, input_feature=list(adata.var_names), device=device,
        latent_dim=n_components, verbose=False,
        # preload_dense=True: concord-sc has a bug where the num_workers kwarg is
        # accepted but never forwarded to its internal DataLoaderManager, so passing
        # num_workers=0 alone does NOT stop it from spawning ~4 worker subprocesses per
        # DataLoader. preload_dense=True hits a different code path in
        # concord/model/dataloader.py that correctly forces num_workers to 0 -- without
        # this, every worker spawn re-imports torch/anndata/concord from scratch, which
        # is fine in a notebook (Jupyter's kernel is __main__, not this file) but turns
        # a single call into tens of seconds of pure import overhead when exported/run
        # as a plain script.
        preload_dense=True,
    )
    cur_ccd.fit_transform(output_key='Concord')
    return adata.obsm['Concord']


def fpca_embed(X, n_components=3):
    generator = syn.FPCADataGenerator()
    generator.fit(X.astype(np.float64), n_fpca_components=n_components)
    return generator.eig_scores


for date in dates:
    print(date)
        
    #Generator = Data_Generator_old(data_path)
    Generator = Data_Generator(data_path)

    N=10000

    graphs=[]
    locations_list=[]
    X_list=[]

    for bbox in bboxes:
        Generator.load_data(bbox=bbox, vars=vars, filename=f'{date}.nc')
        graph = Generator.sample_graph(N=N, k=k, perplexity=perplexity, sampling_method=sampling_method, gen_feature_mode=feature_mode, sigma_method='ones')
        sample = graph.to(device)
        graphs.append(sample)
        locations_list.append(pd.DataFrame({"lat": sample.meta.gps[:,0], "lon": sample.meta.gps[:,1]}))
        X_list.append(sample.normed_values.detach().cpu().numpy())


    df1 = locations_list[0]
    df2 = locations_list[1]

    # Preserve indices as columns
    tmp1 = df1.reset_index().rename(columns={'index': 'idx1'})
    tmp2 = df2.reset_index().rename(columns={'index': 'idx2'})

    # Merge on identical lat/lon pairs
    matches = tmp1.merge(tmp2, on=['lat', 'lon'])

    i1 = matches['idx1'].to_numpy()
    i2 = matches['idx2'].to_numpy()

    #pred_nofe_list = []
    #pred_pca_list = []
    #pred_tsne_list = []
    #pred_umap_list = []
    pred_phate_list = []
    pred_spatialPCA_list = []
    pred_concord_list = []
    pred_fpca_list = []

    for sample in graphs:
        """
        # NOFE
        print('NOFE ...', end='\r')
        pred_nofe = model(sample).detach().cpu().numpy()
        pred_nofe = (pred_nofe - pred_nofe.mean(axis=0)) / pred_nofe.std(axis=0)
        pred_nofe_list.append(pred_nofe)
        """
        
        X = sample.normed_values.detach().cpu().numpy()
        
        """
        # PCA
        print('PCA ...', end='\r')
        pca = PCA(n_components=3)
        pred_pca = pca.fit_transform(X)
        pred_pca = (pred_pca - pred_pca.mean(axis=0)) / pred_pca.std(axis=0)
        pred_pca_list.append(pred_pca)

        # t-SNE
        print('t-SNE ...', end='\r')
        pred_tsne = TSNE(
            n_components=3,
            perplexity=30,
            learning_rate="auto",
            random_state=42
        ).fit_transform(X)
        pred_tsne = (pred_tsne - pred_tsne.mean(axis=0)) / pred_tsne.std(axis=0)
        pred_tsne_list.append(pred_tsne)

        # UMAP
        print('UMAP ...', end='\r')
        umap_id = 'umap_buffer'
        dir = f'{project_root}/ERA5/umap'
        np.save(f'{dir}/{umap_id}.npy', X)
        subprocess.run(f"cd {dir} && conda run -n umap python execute_umap.py {umap_id} > /dev/null 2>&1", shell=True)
        pred_umap = np.load(f'{dir}/res_{umap_id}.npy')
        pred_umap = (pred_umap - pred_umap.mean(axis=0)) / pred_umap.std(axis=0)
        pred_umap_list.append(pred_umap)

        os.remove(f'{dir}/{umap_id}.npy')
        os.remove(f'{dir}/res_{umap_id}.npy')
        """

        # PHATE
        print('PHATE ...', end='\r')
        pred_phate = phate.PHATE(
            n_components=3, knn=15, mds_solver='smacof', random_state=42, verbose=False
        ).fit_transform(X)
        pred_phate = (pred_phate - pred_phate.mean(axis=0)) / pred_phate.std(axis=0)
        pred_phate_list.append(pred_phate)

        # spatialPCA
        print('spatialPCA ...', end='\r')
        coords = sample.normed_coords.detach().cpu().numpy()
        pred_spatialPCA = spatialpca_embed(X, coords, spatialpca_call_id, n_components=3)
        pred_spatialPCA = (pred_spatialPCA - pred_spatialPCA.mean(axis=0)) / pred_spatialPCA.std(axis=0)
        pred_spatialPCA_list.append(pred_spatialPCA)

        # Concord
        print('Concord ...', end='\r')
        pred_concord = concord_embed(X, n_components=3)
        pred_concord = (pred_concord - pred_concord.mean(axis=0)) / pred_concord.std(axis=0)
        pred_concord_list.append(pred_concord)

        # functionalPCA
        print('fPCA ...', end='\r')
        pred_fpca = fpca_embed(X, n_components=3)
        pred_fpca = (pred_fpca - pred_fpca.mean(axis=0)) / pred_fpca.std(axis=0)
        pred_fpca_list.append(pred_fpca)


    #nofe_mean_dist_per_pred.append(np.mean((pred_nofe_list[0][i1] - pred_nofe_list[1][i2])**2, axis=1).mean())
    #pca_mean_dist_per_pred.append(np.mean((pred_pca_list[0][i1] - pred_pca_list[1][i2])**2, axis=1).mean())
    #tsne_mean_dist_per_pred.append(np.mean((pred_tsne_list[0][i1] - pred_tsne_list[1][i2])**2, axis=1).mean())
    #umap_mean_dist_per_pred.append(np.mean((pred_umap_list[0][i1] - pred_umap_list[1][i2])**2, axis=1).mean())
    phate_mean_dist_per_pred.append(np.mean((pred_phate_list[0][i1] - pred_phate_list[1][i2])**2, axis=1).mean())
    spatPCA_mean_dist_per_pred.append(np.mean((pred_spatialPCA_list[0][i1] - pred_spatialPCA_list[1][i2])**2, axis=1).mean())
    concord_mean_dist_per_pred.append(np.mean((pred_concord_list[0][i1] - pred_concord_list[1][i2])**2, axis=1).mean())
    fpca_mean_dist_per_pred.append(np.mean((pred_fpca_list[0][i1] - pred_fpca_list[1][i2])**2, axis=1).mean())


#print(f'NOFE: {np.mean(nofe_mean_dist_per_pred):.3f} $\pm$ {np.std(nofe_mean_dist_per_pred):.3f}')
#print(f'PCA:  {np.mean(pca_mean_dist_per_pred):.3f} $\pm$ {np.std(pca_mean_dist_per_pred):.3f}')
#print(f't-SNE:{np.mean(tsne_mean_dist_per_pred):.3f} $\pm$ {np.std(tsne_mean_dist_per_pred):.3f}')
#print(f'UMAP: {np.mean(umap_mean_dist_per_pred):.3f} $\pm$ {np.std(umap_mean_dist_per_pred):.3f}')
print(f'PHATE: {np.mean(phate_mean_dist_per_pred):.3f} $\pm$ {np.std(phate_mean_dist_per_pred):.3f}')
print(f'spatialPCA: {np.mean(spatPCA_mean_dist_per_pred):.3f} $\pm$ {np.std(spatPCA_mean_dist_per_pred):.3f}')
print(f'Concord: {np.mean(concord_mean_dist_per_pred):.3f} $\pm$ {np.std(concord_mean_dist_per_pred):.3f}')
print(f'fPCA: {np.mean(fpca_mean_dist_per_pred):.3f} $\pm$ {np.std(fpca_mean_dist_per_pred):.3f}')

spatialpca_proc.stdin.write('QUIT\n')
spatialpca_proc.stdin.flush()
spatialpca_proc.wait(timeout=10)

spatialPCA server ready.
2019-01-01


/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:590: RuntimeWarning: divide by zero encountered in matmul
  U = Q @ Uhat
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:590: RuntimeWarning: overflow encountered in matmul
  U = Q @ Uhat
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:590: RuntimeWarning: invalid value encountered in matmul
  U = Q @ Uhat
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:20

concord - WARNING - domain/batch information not found, all samples will be treated as from single domain/batch.
concord - WARNING - Only one domain found in the data. Setting p_intra_domain to 1.0.
Epoch 0 Training


/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch 1 Training
Epoch 2 Training
Epoch 3 Training
Epoch 4 Training
Epoch 5 Training
Epoch 6 Training
Epoch 7 Training
Epoch 8 Training
Epoch 9 Training
Epoch 10 Training
Epoch 11 Training
Epoch 12 Training
Epoch 13 Training
Epoch 14 Training


/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


... (333 outputs left out)

Epoch 1 Training
Epoch 2 Training
Epoch 3 Training
Epoch 4 Training
Epoch 5 Training
Epoch 6 Training
Epoch 7 Training
Epoch 8 Training
Epoch 9 Training
Epoch 10 Training
Epoch 11 Training
Epoch 12 Training
Epoch 13 Training
Epoch 14 Training


/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


PHATE: 1.704 $\pm$ 0.705
spatialPCA: 1.346 $\pm$ 0.718
Concord: 1.432 $\pm$ 0.720
fPCA: 1.602 $\pm$ 0.770


0